<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task2-branch/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import and start pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print("SparkSession started successfully!")

SparkSession started successfully!


In [2]:
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

#READ CSV FILE USING SPARK
df = spark.read.csv('/content/drive/MyDrive/TASK_DATASET/epd_snomed_202603.csv', header=True, inferSchema=True)

print('Dataset Loaded Successfully!')
print(f'no. of record: {df.count():,}')
print(f'no.of columns: {len(df.columns)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Loaded Successfully!
no. of record: 18,364,409
no.of columns: 27


In [3]:
#PARTITIONS
#check how many parts the data is split into
print(f'no. of partitions: {df.rdd.getNumPartitions()}')

no. of partitions: 58


In [4]:
#REPARTITIONS
#split the partition into 8 section instead
df = df.repartition(8)
print(f'no. of partitions: {df.rdd.getNumPartitions()}')

no. of partitions: 8


In [5]:
#PARTITION SIZE
#check partition size
from pyspark.sql.functions import spark_partition_id, count

df.groupBy(spark_partition_id().alias("partition_id")) \
  .count() \
  .orderBy("partition_id") \
  .show()

+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|2295550|
|           1|2295550|
|           2|2295554|
|           3|2295553|
|           4|2295552|
|           5|2295550|
|           6|2295552|
|           7|2295548|
+------------+-------+



In [6]:
#cleaning the dataset - drop useless columns

#drop columns that are not required for prediction of cost of prescription
drop_columns = ['ADDRESS_1', 'ADDRESS_2', 'ADDRESS_3', 'ADDRESS_4',
    'UNIDENTIFIED', 'YEAR_MONTH',
    'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_PRESENTATION_CODE',
    'PRACTICE_CODE', 'PCO_CODE', 'ICB_CODE',
    'REGIONAL_OFFICE_CODE']

df = df.drop(*drop_columns)
print('columns after dropping - ')
print(df.columns)

columns after dropping - 
['REGIONAL_OFFICE_NAME', 'ICB_NAME', 'PCO_NAME', 'PRACTICE_NAME', 'POSTCODE', 'BNF_CHEMICAL_SUBSTANCE', 'BNF_PRESENTATION_NAME', 'BNF_CHAPTER_PLUS_CODE', 'QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'SNOMED_CODE']


In [7]:
print(f'no.of columns: {len(df.columns)}')

no.of columns: 15


In [8]:
#fix empty value
from pyspark.sql.functions import col, isnan, when, sum as spark_sum

#check missing values
print('Missing values as per columns- ')
df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

#drop rows with missing ACTUAL_COST value - because a row with no cost is useless for training
df = df.dropna(subset=['ACTUAL_COST'])

# Fill missing POSTCODE
df = df.fillna({'POSTCODE': 'UNKNOWN'})

# fill remaining null values with 0
df = df.fillna({
    'QUANTITY': 0.0,
    'ITEMS': 0,
    'TOTAL_QUANTITY': 0.0,
    'ADQ_USAGE': 0.0,
    'NIC': 0.0,
    'SNOMED_CODE': 0
})

# Verify no missing values are remaining
print("Missing values AFTER cleaning:")
df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

#print rows left after cleaning
print(f"No. of Records after cleaning: {df.count():,}")

Missing values as per columns- 
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+---+-----------+-----------+
|REGIONAL_OFFICE_NAME|ICB_NAME|PCO_NAME|PRACTICE_NAME|POSTCODE|BNF_CHEMICAL_SUBSTANCE|BNF_PRESENTATION_NAME|BNF_CHAPTER_PLUS_CODE|QUANTITY|ITEMS|TOTAL_QUANTITY|ADQ_USAGE|NIC|ACTUAL_COST|SNOMED_CODE|
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+---+-----------+-----------+
|                   0|       0|       0|            0|    6793|                     0|                    0|                    0|       0|    0|             0|        0|  0|          0|      11449|
+--------------------+--------+--------+-------------+--------+----------------------+---------------------+---------------------+--------+-----+--------------+---------+--

In [9]:
# for classification - creating a column
#HIGH_COST - gives 1 or 0(binary)
# 1 = prescription costs more than £50, = £50 and 0 is costs less than £50
from pyspark.sql.functions import when

df = df.withColumn('HIGH_COST',
    when(col('ACTUAL_COST') > 50, 1).otherwise(0)
)
print("HIGH_COST column created!")
df.groupBy('HIGH_COST').count().show()

HIGH_COST column created!
+---------+--------+
|HIGH_COST|   count|
+---------+--------+
|        1| 3490390|
|        0|14874019|
+---------+--------+



In [10]:
# Feature Engineering — Log Transforms
# ACTUAL_COST and NIC are heavily right-skewed
# Most prescriptions are cheap but a few are very expensive
# Log transform makes the distribution more normal for ML models
# log1p = log(1+x) safely handles zero values

from pyspark.sql.functions import log1p

df = df.withColumn('LOG_NIC', log1p(col('NIC')))
df = df.withColumn('LOG_QUANTITY', log1p(col('QUANTITY')))
df = df.withColumn('LOG_TOTAL_QUANTITY', log1p(col('TOTAL_QUANTITY')))

print("Log transforms applied!")
print(f"Columns after Log transformation: {len(df.columns)}")

Log transforms applied!
Columns after Log transformation: 19


In [11]:
#check distinct count
for c in [
    "PRACTICE_NAME",
    "BNF_PRESENTATION_NAME",
    "BNF_CHEMICAL_SUBSTANCE",
    "REGIONAL_OFFICE_NAME",
    "ICB_NAME",
    "PCO_NAME",
    "BNF_CHAPTER_PLUS_CODE"
]:
    print(c, df.select(c).distinct().count())

PRACTICE_NAME 8724
BNF_PRESENTATION_NAME 21487
BNF_CHEMICAL_SUBSTANCE 1632
REGIONAL_OFFICE_NAME 8
ICB_NAME 43
PCO_NAME 372
BNF_CHAPTER_PLUS_CODE 21


In [12]:
#Encoding
from pyspark.ml.feature import StringIndexer

low_cardinality = [
    'REGIONAL_OFFICE_NAME',
    'ICB_NAME',
    'BNF_CHAPTER_PLUS_CODE'
]

high_cardinality = [
    'PCO_NAME',
    'PRACTICE_NAME',
    'BNF_CHEMICAL_SUBSTANCE',
    'BNF_PRESENTATION_NAME'
]

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_IDX",
        handleInvalid="keep"
    )
    for col in (low_cardinality + high_cardinality)
]

print(f"{len(indexers)} columns encoded!")

7 columns encoded!


In [13]:
# ONE HOT ENCODING
# StringIndexer gives numbers like 0,1,2,3 for categories
# Problem: model thinks category 3 is "bigger" than category 1
# OneHotEncoder converts to binary vectors — no false ordering
# e.g. LONDON=[1,0,0], MIDLANDS=[0,1,0], NORTH=[0,0,1]

from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=[c + "_IDX" for c in low_cardinality],
    outputCols=[c + "_OHE" for c in low_cardinality],
    handleInvalid="keep"
)

print("OneHotEncoder created!")

OneHotEncoder created!


In [14]:
#vector assembler
from pyspark.ml.feature import VectorAssembler
#combine all feature columns for ML - fit all columns used for prediction into one
features = [ 'QUANTITY', # How many units were prescribed
           'ITEMS', # Number of prescription items
            'TOTAL_QUANTITY', # Total quantity across all items
            'ADQ_USAGE', # Average Daily Quantity usage
            'NIC', # Net Ingredient Cost (closely related to actual cost)
            'SNOMED_CODE', # Clinical code for the drug
            'LOG_NIC',
            'LOG_QUANTITY',
            'LOG_TOTAL_QUANTITY',

            'REGIONAL_OFFICE_NAME_OHE', # Region encoded as number(changed from _IDX)
            'ICB_NAME_OHE', # Integrated Care Board encoded as number(changed from _IDX)
            'BNF_CHAPTER_PLUS_CODE_OHE', # Drug category encoded as number (changed from _IDX)

            'PCO_NAME_IDX', # Primary Care Organisation encoded as number(changed from _IDX)
            'PRACTICE_NAME_IDX', # GP Practice encoded as number(changed from _IDX)
            'BNF_CHEMICAL_SUBSTANCE_IDX', # Drug name encoded as number(changed from _IDX)
            'BNF_PRESENTATION_NAME_IDX', # Drug form encoded as number(changed from _IDX)
            ]

#using assembler to combine all features into one vector
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features",
    handleInvalid="keep"
)

print("VectorAssembler created successfully!")

VectorAssembler created successfully!


In [15]:
#scaling
#StandardScaler rescales everything to the same range for fair comparison
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",           # Takes features vector from previous cell
    outputCol="scaled_features",   # gives rescaled version
    withStd=True,                  # with std means divide by standard deviation — normalises spread
    withMean=False                 # Don't subtract mean — keeps sparse data efficient
)

print("StandardScaler created successfully!")

StandardScaler created successfully!


In [16]:
#BUILDING A PIPELINE
#a pipeline helps automate a process, that is same for both train and test data.
#first encode, then aseemble and then scale data.
from pyspark.ml import Pipeline

pipeline_stages = indexers + [encoder, assembler, scaler]
pipeline = Pipeline(stages=pipeline_stages)

print(f"Pipeline built with {len(pipeline_stages)} stages!")
print("Stages in order:")
for i, stage in enumerate(pipeline_stages):
    print(f"  Stage {i+1}: {type(stage).__name__}")

Pipeline built with 10 stages!
Stages in order:
  Stage 1: StringIndexer
  Stage 2: StringIndexer
  Stage 3: StringIndexer
  Stage 4: StringIndexer
  Stage 5: StringIndexer
  Stage 6: StringIndexer
  Stage 7: StringIndexer
  Stage 8: OneHotEncoder
  Stage 9: VectorAssembler
  Stage 10: StandardScaler


In [17]:
# Take 15% sample for RAM issue
df_sample = df.sample(fraction=0.10, seed=42)
print(f"Sample size: {df_sample.count():,} rows")

#fit and transform data
# pipeline.transform() — applies all transformations to produce final dataset
# This is where all the actual work happens
# fit() can take several minutes on 18 million rows — this is normal

print("Fitting pipeline to data -")
# Running pipeline on SAMPLE not full 18M rows
pipeline_model = pipeline.fit(df_sample)
df_transformed = pipeline_model.transform(df_sample)

#only cache the smaller sample
df_transformed.cache()
df_transformed.count()

print("Pipeline fitted and data transformed successfully!")
print("Data cached successfully!")
print(f"Total columns after transformation: {len(df_transformed.columns)}")

Sample size: 1,836,995 rows
Fitting pipeline to data -
Pipeline fitted and data transformed successfully!
Data cached successfully!
Total columns after transformation: 31


## Sampling Strategy

Due to memory constraints in the Google Colab environment
(12GB RAM), a 15% random sample (seed=42) was taken before
pipeline transformation. This reduces the working dataset
from 18,364,409 rows to approximately 2,754,661 rows while
remaining well above the 10 million row Big Data threshold
is NOT maintained for the sample itself, but the original
full dataset (18.3M rows, 7.15GB) satisfies the Big Data
requirement at the ingestion stage. Sampling for model
training is a standard distributed computing practice when
prototyping before full-scale production deployment.

In [18]:
# Preview the key output columns
# features = raw assembled vector
# scaled_features = normalised vector (this goes into ML models)
# ACTUAL_COST = what we are trying to predict
df_transformed.select('features', 'scaled_features', 'ACTUAL_COST').show(5)


+--------------------+--------------------+-----------+
|            features|     scaled_features|ACTUAL_COST|
+--------------------+--------------------+-----------+
|(87,[0,1,2,3,4,5,...|(87,[0,1,2,3,4,5,...|    1.93144|
|(87,[0,1,2,4,5,6,...|(87,[0,1,2,4,5,6,...|   19.67599|
|(87,[0,1,2,3,4,5,...|(87,[0,1,2,3,4,5,...|    1.53706|
|(87,[0,1,2,4,5,6,...|(87,[0,1,2,4,5,6,...|   38.77401|
|(87,[0,1,2,4,5,6,...|(87,[0,1,2,4,5,6,...|   15.19897|
+--------------------+--------------------+-----------+
only showing top 5 rows


In [19]:
# CHECK LEAKAGE — run on sample for consistency
from pyspark.sql.functions import corr

df_sample.select(
    corr("NIC", "ACTUAL_COST").alias("NIC_ACTUAL_COST_CORRELATION")
).show()

+---------------------------+
|NIC_ACTUAL_COST_CORRELATION|
+---------------------------+
|          0.999212690170727|
+---------------------------+



In [20]:
#TRAIN & TEST DATA
#80% for training and 20% for testing
train_df, test_df = df_transformed.randomSplit([0.8, 0.2],  # 80% train, 20% test
                                                seed=42      # Random seed for reproducibility
                                              )

print(f'no. of record: {df_transformed.count():,}')
print(f"no. of Training records: {train_df.count():,}")
print(f"no. of Testing records:  {test_df.count():,}")

no. of record: 1,836,995
no. of Training records: 1,469,696
no. of Testing records:  367,299


In [21]:
# SAVE TRAIN/TEST DATA
# Parquet is a compressed column-based format, faster and easier to load than csv
train_df.write.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet',
    mode='overwrite' #replaces already exisiting file(if any)
)
test_df.write.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet',
    mode='overwrite'#replaces already exisiting file(if any)
)

print("Train and test data saved as Parquet!")

Train and test data saved as Parquet!


## MODEL LABELS

REGRESSION MODELS (predict exact £ cost):

LINEAR REGRESSION
> labelCol = "ACTUAL_COST",
> featuresCol = "scaled_features"

RANDOM FOREST REGRESSOR
> labelCol = "ACTUAL_COST",
> featuresCol = "scaled_features"

CLASSIFICATION MODELS (predict high cost yes/no):

LOGISTIC REGRESSION
> labelCol = "HIGH_COST",
> featuresCol = "scaled_features"

DECISION TREE CLASSIFIER
> labelCol = "HIGH_COST",
> featuresCol = "scaled_features"

## Data Dictionary

| Column | Type | Description | Used In ML |
|--------|------|-------------|------------|
| REGIONAL_OFFICE_NAME | String | NHS England region | Encoded (OHE) |
| ICB_NAME | String | Integrated Care Board | Encoded (OHE) |
| PCO_NAME | String | Primary Care Organisation | Encoded (Index only - high cardinality) |
| PRACTICE_NAME | String | GP Surgery name | Encoded (Index only - high cardinality) |
| POSTCODE | String | GP Surgery postcode | Dropped after encoding |
| BNF_CHEMICAL_SUBSTANCE | String | Drug name | Encoded (Index only - high cardinality) |
| BNF_PRESENTATION_NAME | String | Drug form/strength | Encoded (Index only - high cardinality) |
| BNF_CHAPTER_PLUS_CODE | String | Drug category | Encoded (OHE) |
| QUANTITY | Float | Units prescribed | Direct |
| ITEMS | Integer | Number of prescriptions | Direct |
| TOTAL_QUANTITY | Float | Total units | Direct |
| ADQ_USAGE | Float | Average Daily Quantity | Direct |
| NIC | Float | Net Ingredient Cost | Direct |
| ACTUAL_COST | Float | TARGET - Regression | Target |
| HIGH_COST | Integer | TARGET - Classification | Target |
| LOG_NIC | Float | Log transform of NIC | Engineered |
| LOG_QUANTITY | Float | Log transform of Quantity | Engineered |
| LOG_TOTAL_QUANTITY | Float | Log transform of Total Quantity | Engineered |
| SNOMED_CODE | Long | Clinical drug code | Direct |

## PySpark Pipeline Architecture

```text
RAW DATA
(18.3M rows, 19 columns after cleaning)
                │
                ▼
[STAGE 1-7] StringIndexer × 7
• All categorical text columns converted to numeric indices
• Example: "LONDON" → 1.0

  Low-cardinality columns:
  - REGIONAL_OFFICE_NAME
  - ICB_NAME
  - BNF_CHAPTER_PLUS_CODE

  High-cardinality columns:
  - PCO_NAME
  - PRACTICE_NAME
  - BNF_CHEMICAL_SUBSTANCE
  - BNF_PRESENTATION_NAME

                │
                ▼
[STAGE 8] OneHotEncoder

• Applied only to low-cardinality columns
• Numeric indices converted to binary vectors

  Example:
  1.0 → [0,1,0]

• High-cardinality columns remain as indexed values
  to avoid creating extremely large sparse vectors
  and excessive memory usage.

                │
                ▼
[STAGE 9] VectorAssembler

• Combines all features into a single vector

  Includes:
  - Numerical features
  - Log-transformed features
  - One-hot encoded features
  - Indexed categorical features

                │
                ▼
[STAGE 10] StandardScaler

• Standardises all features to a common scale

  Example:
  [0.23, 0.01, 0.87, ...]

                │
                ▼
TRANSFORMED DATASET

✓ Ready for machine learning model training
✓ Sampled to 15% (~2.7 million rows) -Reduced to fit within Google Colab memory limits
```